In [0]:
import dlt
from pyspark.sql.functions import *

dlt.create_streaming_table("electroniz_silver_customers")

In [0]:
@dlt.view
@dlt.expect_or_drop("valid_credit_card", "credit_card IS NOT NULL")
@dlt.expect_or_drop("valid_email", "email RLIKE '^[a-zA-Z0-9._%+\\\\-]+@[a-zA-Z0-9.\\\\-]+\\\\.[a-zA-Z]{2,}$'")
def v_clean_customers():
    bronze = spark.readStream.table("electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_store_customers")
    return bronze.select(
        col("customer_id"),
        col("customer_name"),
        col("address"),
        col("city"),
        col("postalcode"),
        when(upper(col("country")).isin("UNITED STATES", "US", "U.S.", "AMERICA"), "US")
        .when(upper(col("country")).isin("UNITED KINGDOM", "U.K"), "UK")
        .when(upper(col("country")).isin("INDIA", "IN"), "IND")
        .when(upper(col("country")).isin("CANADA", "CA"), "CAN")
        .otherwise(col("country")).alias("country"),
        col("phone"),
        col("email"),
        col("credit_card"),
        col("updated_at")
    )

In [0]:
dlt.apply_changes(
    target="electroniz_silver_customers",
    source="v_clean_customers",
    keys=["customer_id"],
    sequence_by=col("updated_at"),
    stored_as_scd_type=2
)